In [1]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [2]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import time
import psutil

import pandas as pd

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama

/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_35407/799256754.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [4]:
# ============================================================
# LOAD OPENAI EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [5]:
# ============================================================
# LOAD FAISS VECTOR STORE
# ============================================================

vector_store = FAISS.load_local(

    "faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [6]:
# ============================================================
# TEST FAISS RETRIEVAL
# ============================================================

docs = vector_store.similarity_search(

    "Harry Potter actor",

    k=2

)

print(
    "Documents Retrieved:",
    len(docs)
)

Documents Retrieved: 2


In [7]:
# ============================================================
# LOAD MISTRAL MODEL
# ============================================================

llm = ChatOllama(

    model="mistral",

    temperature=0

)

print(
    "Mistral Loaded Successfully"
)

Mistral Loaded Successfully


In [8]:
# ============================================================
# TEST MISTRAL CONNECTION
# ============================================================

response = llm.invoke(
    "Say Hello"
)

print(
    response.content
)

 Hello there! How can I assist you today?


In [9]:
# ============================================================
# EVALUATION QUERY SET
# ============================================================

queries = [

    "Harry Potter actor",

    "World Cup winner",

    "US election",

    "climate change",

    "Olympics",

    "stock market",

    "global warming",

    "terrorism",

    "education reforms",

    "covid vaccine",

    "space exploration",

    "economic crisis",

    "oil prices",

    "artificial intelligence",

    "healthcare policy",

    "renewable energy",

    "sports championship",

    "government budget",

    "scientific discovery",

    "international relations"
]

print(
    "Queries Loaded:",
    len(queries)
)

Queries Loaded: 20


In [10]:
# ============================================================
# RESULTS CONTAINER
# ============================================================

results = []

In [11]:
# ============================================================
# DENSE RAG PIPELINE
# ============================================================

for query in queries:

    print(
        f"\nProcessing Query: {query}"
    )

    start_time = time.time()

    memory_before = (

        psutil.Process()
        .memory_info()
        .rss

        /

        1024

        /

        1024

    )

    # --------------------------------------------------------
    # RETRIEVE DOCUMENTS
    # --------------------------------------------------------

    docs = vector_store.similarity_search(

        query,

        k=5

    )

    context = "\n\n".join(

        [
            doc.page_content

            for doc in docs
        ]

    )

    print(
        "Retrieved Documents:",
        len(docs)
    )

    print(
        "Context Length:",
        len(context)
    )

    # --------------------------------------------------------
    # CREATE PROMPT
    # --------------------------------------------------------

    prompt = f"""

    You are a helpful AI assistant.

    Use ONLY the provided context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # --------------------------------------------------------
    # GENERATE RESPONSE
    # --------------------------------------------------------

    response = llm.invoke(
        prompt
    )

    answer = response.content

    latency = (

        time.time()

        -

        start_time

    )

    memory_after = (

        psutil.Process()
        .memory_info()
        .rss

        /

        1024

        /

        1024

    )

    memory_used = (

        memory_after

        -

        memory_before

    )

    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    results.append({

        "pipeline":
        "Dense Mistral",

        "retrieval_method":
        "Dense",

        "model":
        "Mistral",

        "query":
        query,

        "retrieved_count":
        len(docs),

        "retrieved_docs":
        str(
            [
                doc.page_content[:500]

                for doc in docs
            ]
        ),

        "context":
        context,

        "context_length":
        len(context),

        "generated_answer":
        answer,

        "response_length":
        len(answer),

        "latency_seconds":
        latency,

        "memory_mb":
        memory_used

    })

print(
    "\nDense Mistral Pipeline Completed"
)


Processing Query: Harry Potter actor
Retrieved Documents: 5
Context Length: 2160

Processing Query: World Cup winner
Retrieved Documents: 5
Context Length: 2002

Processing Query: US election
Retrieved Documents: 5
Context Length: 2494

Processing Query: climate change
Retrieved Documents: 5
Context Length: 2043

Processing Query: Olympics
Retrieved Documents: 5
Context Length: 2483

Processing Query: stock market
Retrieved Documents: 5
Context Length: 2498

Processing Query: global warming
Retrieved Documents: 5
Context Length: 2486

Processing Query: terrorism
Retrieved Documents: 5
Context Length: 1990

Processing Query: education reforms
Retrieved Documents: 5
Context Length: 1473

Processing Query: covid vaccine
Retrieved Documents: 5
Context Length: 2054

Processing Query: space exploration
Retrieved Documents: 5
Context Length: 1960

Processing Query: economic crisis
Retrieved Documents: 5
Context Length: 2077

Processing Query: oil prices
Retrieved Documents: 5
Context Length:

In [12]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 20


,pipeline,retrieval_method,model,query,retrieved_count,retrieved_docs,context,context_length,generated_answer,response_length,latency_seconds,memory_mb
0,Dense Mistral,Dense,Mistral,Harry Potter actor,5,['malfoy rupert grint ron weasley and director...,malfoy rupert grint ron weasley and director d...,2160,"Daniel Radcliffe, who plays Harry Potter.",42,16.401700,943.546875
1,Dense Mistral,Dense,Mistral,World Cup winner,5,['world cups including this summers tournament...,world cups including this summers tournament i...,2002,The World Cup winner mentioned in the context...,428,29.028848,6.890625
2,Dense Mistral,Dense,Mistral,US election,5,['president since 1964 going into the election...,president since 1964 going into the election n...,2494,The US election in question is the presidenti...,460,28.551036,-2.765625
3,Dense Mistral,Dense,Mistral,climate change,5,['climate change let us know in the sound off ...,climate change let us know in the sound off bo...,2043,Climate change is happening faster than previ...,327,22.170053,1.171875
4,Dense Mistral,Dense,Mistral,Olympics,5,['one another thats what the olympics are abou...,one another thats what the olympics are about ...,2483,The Olympics are events that symbolize greatn...,386,26.041477,-1.531250


In [13]:
# ============================================================
# VALIDATE DATAFRAME
# ============================================================

results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pipeline          20 non-null     str    
 1   retrieval_method  20 non-null     str    
 2   model             20 non-null     str    
 3   query             20 non-null     str    
 4   retrieved_count   20 non-null     int64  
 5   retrieved_docs    20 non-null     str    
 6   context           20 non-null     str    
 7   context_length    20 non-null     int64  
 8   generated_answer  20 non-null     str    
 9   response_length   20 non-null     int64  
 10  latency_seconds   20 non-null     float64
 11  memory_mb         20 non-null     float64
dtypes: float64(2), int64(3), str(7)
memory usage: 99.8 KB


In [14]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(

    "dense_mistral_results.csv",

    index=False

)

print(
    "Dense Mistral Results Saved Successfully"
)

Dense Mistral Results Saved Successfully
